In [1]:
tabla='hmese10'
dim='dim_estenf'

# COLUMNAS LAKE

In [2]:
import re

cadena = """
    oricenasicod character(1) COLLATE pg_catalog."default",
    cenasicod character(3) COLLATE pg_catalog."default",
    arehoscod character(2) COLLATE pg_catalog."default",
    servhoscod character(3) COLLATE pg_catalog."default",
    estenfcod character(2) COLLATE pg_catalog."default",
    estenfdes character(20) COLLATE pg_catalog."default",
    estenfubides character(20) COLLATE pg_catalog."default",
    estenfestregcod character(1) COLLATE pg_catalog."default",
    estenfcamflg character(1) COLLATE pg_catalog."default"
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ','.join(nombrecitos)

print(insertado_col) 

ALTER COLUMN oricenasicod TYPE character(1),
ALTER COLUMN cenasicod TYPE character(3),
ALTER COLUMN arehoscod TYPE character(2),
ALTER COLUMN servhoscod TYPE character(3),
ALTER COLUMN estenfcod TYPE character(2),
ALTER COLUMN estenfdes TYPE character(20),
ALTER COLUMN estenfubides TYPE character(20),
ALTER COLUMN estenfestregcod TYPE character(1),
ALTER COLUMN estenfcamflg TYPE character(1);

oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,estenfdes,estenfubides,estenfestregcod,estenfcamflg


In [3]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [4]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527
engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)
connection2.close()

In [5]:
base2.head()

,oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,estenfdes,estenfubides,estenfestregcod,estenfcamflg,estenftopemerel
0,1,076,03,AB1,01,MEDICINA GENERAL,,1,None,None
1,1,076,03,D14,05,NEONATOLOGIA,SEGUNDO PISO,1,1,None
2,1,076,03,AK1,06,UCI,SEGUNDO PISO,1,1,None
3,1,308,03,AB1,06,HGEN COVID 19,2DO. PISO,1,1,None
4,1,265,03,F21,03,HOSP. GINECO OBS,GINECO OBS,0,None,None


In [6]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'arehoscod', 'servhoscod', 'estenfcod',
       'estenfdes', 'estenfubides', 'estenfestregcod', 'estenfcamflg',
       'estenftopemerel'],
      dtype='object')

In [7]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")



Cantidad de filas en la tabla hmese10 antes de la inserción: 4024


In [8]:
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

145

In [9]:
#ALTER SEQUENCE dim_consult_id_consult_seq RESTART WITH 0;

In [10]:


query=f"""
ALTER TABLE tmp_{tabla}
{cadena_alter}


UPDATE {tabla} b

SET 
estenfdes=t.estenfdes,
estenfubides=t.estenfubides,
estenfestregcod=t.estenfestregcod,
estenfcamflg=t.estenfcamflg

FROM tmp_{tabla} t 
WHERE 

b.oricenasicod = t.oricenasicod and b.oricenasicod IS NOT NULL AND
b.cenasicod = t.cenasicod and b.cenasicod IS NOT NULL AND
b.arehoscod = t.arehoscod and b.arehoscod IS NOT NULL AND
b.servhoscod = t.servhoscod and b.servhoscod IS NOT NULL AND
b.estenfcod = t.estenfcod and b.estenfcod IS NOT NULL ;


INSERT INTO {tabla} 
({insertado_col}) 

SELECT 
{insertado_col}

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {tabla} b 
    WHERE 
    
    b.oricenasicod = t.oricenasicod and b.oricenasicod IS NOT NULL AND
    b.cenasicod = t.cenasicod and b.cenasicod IS NOT NULL AND
    b.arehoscod = t.arehoscod and b.arehoscod IS NOT NULL AND
    b.servhoscod = t.servhoscod and b.servhoscod IS NOT NULL AND
    b.estenfcod = t.estenfcod and b.estenfcod IS NOT NULL 

) ;


DROP TABLE tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [11]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla hmese10 después de la inserción: 4145
La cantidad de filas insertadas fue de: 121


In [12]:
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)

In [13]:
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)


145

In [14]:
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")


Cantidad de filas en la tabla dim_estenf antes de la inserción: 4050


In [15]:

query=f"""

ALTER TABLE tmp_{tabla} 
{cadena_alter}
INSERT INTO {dim} 
(ori_cas,cod_cas,are_cod,ser_cod,est_cod,est_des,est_ubi) 

SELECT 

oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,estenfdes,estenfubides

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.ori_cas = t.oricenasicod AND
    d.cod_cas = t.cenasicod AND
    d.are_cod = t.arehoscod AND
    d.ser_cod = t.servhoscod AND
    d.est_cod = t.estenfcod
);

DROP TABLE tmp_{tabla};
"""

c1= text(query)
cursor=connection4.execute(c1)



In [16]:

query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


Cantidad de filas en la tabla hmese10 después de la inserción: 4145
La cantidad de filas insertadas fue de: 95


In [17]:
connection4.close()
connection3.close()